In [1]:
print("hola mundo")

hola mundo


In [3]:
!pip install mysql-connector-python pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 1.0 MB/s eta 0:00:0000:0100:01m


In [20]:
import mysql.connector
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [17]:
# Configuracion de conexion

DB_CONFIG = {
    'host':     '127.0.0.1',   # o la IP de tu servidor MySQL
    'port': '10101',
    'user':     'root',        # tu usuario de MySQL
    'password': 'admin123', # tu contraseña
}

# Nombres de las tres tablas que deseas analizar
DATABASES = [
    'pia_data_01',
    'pia_data_02',
    'pia_data_03'
]

In [23]:

def get_connection(host: str, port: str, user: str, password: str, database: str):
    """
    Crea y retorna un motor (engine) de SQLAlchemy conectado a MySQL.
    """
    # Creamos la cadena de conexión (URI)
    # Usamos mysql+mysqlconnector para aprovechar la librería que ya usabas
    conexion_uri = f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{database}"
    
    try:
        # create_engine crea el motor de conexión
        engine = create_engine(conexion_uri)
        
        # Probamos la conexión abriéndola y cerrándola rápidamente
        with engine.connect() as conn:
            print(f'Conexión exitosa a [{database}]')
            
        return engine
    except SQLAlchemyError as e:
        print(f' Error de conexión: {e}')
        return None

def get_raw_data(engine) -> pd.DataFrame:
    """
    Extrae todos los datos de una tabla MySQL usando SQLAlchemy.
    """
    try:
        query = f'SELECT * FROM salaries'
        # Pandas recibe felizmente el motor de SQLAlchemy
        df = pd.read_sql(query, con=engine)
        print(f'  📥 [salaries] → {df.shape[0]:,} filas × {df.shape[1]} columnas')
        return df
    except Exception as e:
        print(f' Error al leer [{table_name}]: {e}')
        return pd.DataFrame()

def close_connection(engine):
    """
    Cierra todas las conexiones activas en el pool y desecha el motor.
    Úsalo al final de tu script o aplicación.
    """
    if engine:
        try:
            engine.dispose()
            print("🔌 Conexiones de SQLAlchemy cerradas y recursos liberados con éxito.")
        except Exception as e:
            print(f" Error al cerrar el motor de SQLAlchemy: {e}")

# ── Ejecutar extracción ─────────────────────────────────────────────────────
conn_01 = get_connection(**DB_CONFIG, database=DATABASES[0])
conn_02 = get_connection(**DB_CONFIG, database=DATABASES[1])
conn_03 = get_connection(**DB_CONFIG, database=DATABASES[2])
if conn:
    print('\n Extrayendo datos de las tablas...')
    data_raw_01 = get_raw_data(conn_01)
    data_raw_02 = get_raw_data(conn_02)
    data_raw_03 = get_raw_data(conn_03)

    
    close_connection(conn_01)
    close_connection(conn_02)
    close_connection(conn_03)
    print('\n Conexión cerrada.')
else:
    print('  No se pudo establecer conexión. Verifica DB_CONFIG.')

Conexión exitosa a [pia_data_01]
Conexión exitosa a [pia_data_02]
Conexión exitosa a [pia_data_03]

 Extrayendo datos de las tablas...
  📥 [salaries] → 710 filas × 15 columnas
  📥 [salaries] → 1,050 filas × 15 columnas
  📥 [salaries] → 2,333 filas × 15 columnas
🔌 Conexiones de SQLAlchemy cerradas y recursos liberados con éxito.
🔌 Conexiones de SQLAlchemy cerradas y recursos liberados con éxito.
🔌 Conexiones de SQLAlchemy cerradas y recursos liberados con éxito.

 Conexión cerrada.


In [24]:
#Previsualizacion de dataset
def preview_dataset(df: pd.DataFrame, table_name: str, n_rows: int = 5):
    """Muestra una vista rápida del dataset."""
    print(f'\n{'='*60}')
    print(f'  TABLE: {table_name}')
    print(f'{'='*60}')
    print(f'Shape: {df.shape}')
    print(f'\nDtypes:\n{df.dtypes}')
    print(f'\nPrimeras {n_rows} filas:')
    display(df.head(n_rows))
    print(f'\nEstadísticas descriptivas:')
    display(df.describe(include='all').T)

for df_i, name_i in zip([data_raw_01, data_raw_02, data_raw_03], TABLE_NAMES):
    if not df_i.empty:
        preview_dataset(df_i, name_i)


  TABLE: pia_data_01
Shape: (710, 15)

Dtypes:
id                          int64
gender                     object
age                       float64
country                    object
married                     int64
family_members              int64
industry                   object
income_flux               float64
outgoing_flux             float64
education_level            object
years_of_experience         int64
employment_type            object
remote_work_percentage      int64
currency                   object
salary_satisfaction         int64
dtype: object

Primeras 5 filas:


,id,gender,age,country,married,family_members,industry,income_flux,outgoing_flux,education_level,years_of_experience,employment_type,remote_work_percentage,currency,salary_satisfaction
0,1,Female,30.0,Guatemala,0,3,Pharmaceutical,0.0,3350.0,Other,7,Unemployed,0,GTQ,3
1,2,Female,37.0,Guatemala,0,1,Pharmaceutical,14380.0,9343.0,Master,10,Full-time,32,GTQ,5
2,3,Female,59.0,Guatemala,1,2,Health,14282.0,9239.0,Bachelor,22,Unemployed,0,GTQ,3
3,4,Female,38.0,Guatemala,1,0,Economics,12862.0,7010.0,Master,11,None,19,GTQ,3
4,5,Male,31.0,Guatemala,0,4,Technology,2057.0,693.0,Bachelor,4,Freelance,42,GTQ,2



Estadísticas descriptivas:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,710.0,NaN,NaN,NaN,355.5,205.103632,1.0,178.25,355.5,532.75,710.0
gender,675,2,Female,370,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,696.0,NaN,NaN,NaN,40.922414,13.905012,18.0,28.0,40.0,54.0,64.0
country,710,1,Guatemala,710,NaN,NaN,NaN,NaN,NaN,NaN,NaN
married,710.0,NaN,NaN,NaN,0.521127,0.499906,0.0,0.0,1.0,1.0,1.0
family_members,710.0,NaN,NaN,NaN,2.502817,1.700201,0.0,1.0,2.0,4.0,5.0
industry,710,8,Agriculture,106,NaN,NaN,NaN,NaN,NaN,NaN,NaN
income_flux,684.0,NaN,NaN,NaN,7451.678363,6825.56772,0.0,1000.0,6076.0,12173.5,30466.0
outgoing_flux,675.0,NaN,NaN,NaN,3843.85037,3243.610036,201.0,1091.5,3190.0,5669.0,17867.0
education_level,689,5,Bachelor,343,NaN,NaN,NaN,NaN,NaN,NaN,NaN



  TABLE: pia_data_02
Shape: (1050, 15)

Dtypes:
id                          int64
gender                     object
age                       float64
country                    object
married                     int64
family_members              int64
industry                   object
income_flux               float64
outgoing_flux             float64
education_level            object
years_of_experience         int64
employment_type            object
remote_work_percentage      int64
currency                   object
salary_satisfaction         int64
dtype: object

Primeras 5 filas:


...

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,1050.0,NaN,NaN,NaN,525.5,303.253195,1.0,263.25,525.5,787.75,1050.0
gender,998,2,Female,512,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,1029.0,NaN,NaN,NaN,40.165209,13.541761,18.0,28.0,40.0,52.0,64.0
country,1050,1,USA,1050,NaN,NaN,NaN,NaN,NaN,NaN,NaN
married,1050.0,NaN,NaN,NaN,0.500952,0.500237,0.0,0.0,1.0,1.0,1.0
family_members,1050.0,NaN,NaN,NaN,2.598095,1.708052,0.0,1.0,3.0,4.0,5.0
industry,1050,10,Health,120,NaN,NaN,NaN,NaN,NaN,NaN,NaN
income_flux,1010.0,NaN,NaN,NaN,43385.168317,24968.501944,0.0,28413.75,45911.0,60209.25,200000.0
outgoing_flux,998.0,NaN,NaN,NaN,22482.189379,11583.079182,208.0,13550.25,21013.5,29673.5,66881.0
education_level,1019,5,Bachelor,488,NaN,NaN,NaN,NaN,NaN,NaN,NaN



  TABLE: pia_data_03
Shape: (2333, 15)

Dtypes:
id                          int64
gender                     object
age                       float64
country                    object
married                     int64
family_members              int64
industry                   object
income_flux               float64
outgoing_flux             float64
education_level            object
years_of_experience         int64
employment_type            object
remote_work_percentage      int64
currency                   object
salary_satisfaction         int64
dtype: object

Primeras 5 filas:


,id,gender,age,country,married,family_members,industry,income_flux,outgoing_flux,education_level,years_of_experience,employment_type,remote_work_percentage,currency,salary_satisfaction
0,1,Male,42.0,Brazil,1,5,Technology,200000.0,111671.0,High School,15,Full-time,0,BRL,5
1,2,Male,61.0,Japan,0,0,Food,200000.0,55909.0,Bachelor,18,Full-time,54,JPY,3
2,3,None,NaN,China,0,5,Law,23124.0,125946.0,Bachelor,5,Unemployed,0,CNY,4
3,4,Female,63.0,Japan,1,4,IoT,200000.0,93318.0,Bachelor,23,Freelance,26,JPY,2
4,5,Male,40.0,India,1,5,Manufacture,200000.0,99466.0,Bachelor,10,Part-time,63,INR,3



Estadísticas descriptivas:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,2333.0,NaN,NaN,NaN,1167.0,673.623411,1.0,584.0,1167.0,1750.0,2333.0
gender,2217,2,Male,1117,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,2287.0,NaN,NaN,NaN,40.58286,13.423515,18.0,29.0,40.0,52.0,64.0
country,2333,8,Australia,314,NaN,NaN,NaN,NaN,NaN,NaN,NaN
married,2333.0,NaN,NaN,NaN,0.518217,0.499775,0.0,0.0,1.0,1.0,1.0
family_members,2333.0,NaN,NaN,NaN,2.481783,1.678294,0.0,1.0,2.0,4.0,5.0
industry,2333,10,Law,266,NaN,NaN,NaN,NaN,NaN,NaN,NaN
income_flux,2246.0,NaN,NaN,NaN,100022.83927,78904.140405,0.0,34723.75,68775.5,200000.0,200000.0
outgoing_flux,2217.0,NaN,NaN,NaN,56333.317095,39047.930109,238.0,22451.0,46697.0,88361.0,139901.0
education_level,2264,5,Bachelor,1168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
